In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

file_path = '/content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/entity-canonicalization/final_canonicalized_LOGIC_FILTERED_FULL.csv'

# Đọc file CSV
try:
    df_final = pd.read_csv(file_path, encoding='utf-8-sig')

    print("Đã đọc file JSON vào DataFrame thành công!")
    print("5 dòng dữ liệu đầu tiên:")
    print(df_final.head())

except Exception as e:
    print(f"Đã xảy ra lỗi khi đọc file bằng pandas: {e}")

Đã đọc file JSON vào DataFrame thành công!
5 dòng dữ liệu đầu tiên:
  head_entity_id    head_entity_text head_entity_type tail_entity_id  \
0      ent_00002          viêm gan B    DISEASESYMTOM      ent_00354   
1      ent_00002          viêm gan B    DISEASESYMTOM      ent_10703   
2      ent_10703  sung huyết hang vị    DISEASESYMTOM      ent_00354   
3      ent_00949        cao huyết áp    DISEASESYMTOM      ent_00340   
4      ent_00237            mãn tính    DISEASESYMTOM      ent_01241   

     tail_entity_text tail_entity_type relation_type  confidence source_type  \
0   trào ngược dạ dày    DISEASESYMTOM   HAS_SYMPTOM    0.999892    question   
1  sung huyết hang vị    DISEASESYMTOM   HAS_SYMPTOM    0.999892    question   
2   trào ngược dạ dày    DISEASESYMTOM   HAS_SYMPTOM    0.999892    question   
3          viêm gan C    DISEASESYMTOM   HAS_SYMPTOM    0.999892    question   
4          nổi mề đay    DISEASESYMTOM   HAS_SYMPTOM    0.999892    question   

                  

In [4]:
import pandas as pd
import numpy as np

# Chọn 3 cột chính
triple_df = df_final[['head_entity_text_canonical',
                      'relation_type',
                      'tail_entity_text_canonical']].copy()

# Đổi tên cột sang định dạng S-P-O chuẩn
triple_df.columns = ['subject', 'predicate', 'object']

print("5 Triple đầu tiên:")
print(triple_df.head())

5 Triple đầu tiên:
              subject    predicate              object
0          VIÊM GAN B  HAS_SYMPTOM   TRÀO NGƯỢC DẠ DÀY
1          VIÊM GAN B  HAS_SYMPTOM  XUNG HUYẾT HANG VỊ
2  XUNG HUYẾT HANG VỊ  HAS_SYMPTOM   TRÀO NGƯỢC DẠ DÀY
3        CAO HUYẾT ÁP  HAS_SYMPTOM          VIÊM GAN C
4            MÃN TÍNH  HAS_SYMPTOM          NỔI MỀ ĐAY


In [5]:
# --- Lưu dưới dạng Text File (phân tách bằng tab) ---
TEXT_FILE_PATH = '/content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/knowledge_graph_triples.txt'

try:
    # index=False để không ghi số thứ tự dòng
    # header=False để không ghi tên cột
    triple_df.to_csv(
        TEXT_FILE_PATH,
        sep='\t',
        index=False,
        header=False,
        encoding='utf-8'
    )
    print(f"\n🎉 Đã lưu Triple-format dưới dạng TEXT FILE thành công tới: {TEXT_FILE_PATH}")
except Exception as e:
    print(f"LỖI khi lưu Text File: {e}")


🎉 Đã lưu Triple-format dưới dạng TEXT FILE thành công tới: /content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/knowledge_graph_triples.txt


In [6]:
# --- Lưu dưới dạng CSV ---
CSV_FILE_PATH = '/content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/knowledge_graph_triples.csv'

try:
    triple_df.to_csv(
        CSV_FILE_PATH,
        index=False,
        encoding='utf-8-sig' # utf-8-sig phù hợp với Excel trên Windows
    )
    print(f"\n🎉 Đã lưu Triple-format dưới dạng CSV thành công tới: {CSV_FILE_PATH}")
except Exception as e:
    print(f"LỖI khi lưu CSV: {e}")


🎉 Đã lưu Triple-format dưới dạng CSV thành công tới: /content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/knowledge_graph_triples.csv


In [7]:
from sklearn.model_selection import train_test_split
import os

# --- Cấu hình ---
# Tỷ lệ: 80% Train, 10% Validation, 10% Test
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Phân chia thành Train và Temp (Validation + Test)
df_train, df_temp = train_test_split(
    triple_df,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=42 # Đảm bảo kết quả chia là cố định
)

# Phân chia Temp thành Validation và Test
df_val, df_test = train_test_split(
    df_temp,
    test_size=(TEST_RATIO / (VAL_RATIO + TEST_RATIO)),
    random_state=42
)

# --- Lưu 3 tập dữ liệu theo định dạng PyKEEN ---
BASE_PATH = '/content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/pykeen_dataset/'
os.makedirs(BASE_PATH, exist_ok=True)

df_train.to_csv(os.path.join(BASE_PATH, 'training.tsv'), sep='\t', index=False, header=False)
df_val.to_csv(os.path.join(BASE_PATH, 'validation.tsv'), sep='\t', index=False, header=False)
df_test.to_csv(os.path.join(BASE_PATH, 'testing.tsv'), sep='\t', index=False, header=False)

print("\n--- Định dạng PyKEEN Dataset ---")
print(f"🎉 Đã lưu 3 tập Train/Validation/Test cho PyKEEN tại thư mục: {BASE_PATH}")
print(f"Train: {len(df_train)}, Validation: {len(df_val)}, Test: {len(df_test)}")


--- Định dạng PyKEEN Dataset ---
🎉 Đã lưu 3 tập Train/Validation/Test cho PyKEEN tại thư mục: /content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/pykeen_dataset/
Train: 24764, Validation: 3095, Test: 3096


In [8]:
# 1. Làm sạch DataFrame cuối cùng trước khi chia
# Loại bỏ các hàng có bất kỳ giá trị nào là NaN hoặc chuỗi rỗng
triple_df = df_final[['head_entity_text_canonical', 'relation_type', 'tail_entity_text_canonical']].copy()
triple_df.columns = ['subject', 'predicate', 'object']

# Chuyển về kiểu chuỗi và loại bỏ khoảng trắng dư thừa
triple_df['subject'] = triple_df['subject'].astype(str).str.strip()
triple_df['predicate'] = triple_df['predicate'].astype(str).str.strip()
triple_df['object'] = triple_df['object'].astype(str).str.strip()

# Lọc bỏ các hàng có bất kỳ giá trị nào bị coi là 'rỗng' hoặc 'NaN'
triple_df = triple_df[
    (triple_df['subject'].str.len() > 1) &
    (triple_df['object'].str.len() > 1) &
    (triple_df['predicate'].str.len() > 1)
]

# 2. CHIA TẬP VÀ LƯU FILE TSV (Tái sử dụng code cũ)
from sklearn.model_selection import train_test_split
import os

# --- Cấu hình ---
# Tỷ lệ: 80% Train, 10% Validation, 10% Test
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Phân chia thành Train và Temp (Validation + Test)
df_train, df_temp = train_test_split(
    triple_df,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=42 # Đảm bảo kết quả chia là cố định
)

# Phân chia Temp thành Validation và Test
df_val, df_test = train_test_split(
    df_temp,
    test_size=(TEST_RATIO / (VAL_RATIO + TEST_RATIO)),
    random_state=42
)

# --- Lưu 3 tập dữ liệu theo định dạng PyKEEN ---
BASE_PATH = '/content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/pykeen_dataset_new/'
os.makedirs(BASE_PATH, exist_ok=True)

df_train.to_csv(os.path.join(BASE_PATH, 'training.tsv'), sep='\t', index=False, header=False)
df_val.to_csv(os.path.join(BASE_PATH, 'validation.tsv'), sep='\t', index=False, header=False)
df_test.to_csv(os.path.join(BASE_PATH, 'testing.tsv'), sep='\t', index=False, header=False)

print("\n--- Định dạng PyKEEN Dataset ---")
print(f"🎉 Đã lưu 3 tập Train/Validation/Test cho PyKEEN tại thư mục: {BASE_PATH}")
print(f"Train: {len(df_train)}, Validation: {len(df_val)}, Test: {len(df_test)}")


--- Định dạng PyKEEN Dataset ---
🎉 Đã lưu 3 tập Train/Validation/Test cho PyKEEN tại thư mục: /content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/pykeen_dataset_new/
Train: 24764, Validation: 3095, Test: 3096


In [9]:
# 1. Trích xuất tất cả các thực thể độc nhất
all_heads = df_final[['head_entity_text_canonical', 'head_entity_type']].rename(
    columns={'head_entity_text_canonical': 'id', 'head_entity_type': 'label'}
)
all_tails = df_final[['tail_entity_text_canonical', 'tail_entity_type']].rename(
    columns={'tail_entity_text_canonical': 'id', 'tail_entity_type': 'label'}
)

# 2. Kết hợp và loại bỏ trùng lặp (dựa trên id)
nodes_df = pd.concat([all_heads, all_tails]).drop_duplicates(subset=['id']).reset_index(drop=True)

# 3. Thêm cột ID nội bộ cho Neo4j (Tùy chọn)
# nodes_df[':ID'] = 'e' + (nodes_df.index + 1).astype(str)
nodes_df.columns = ['name:ID', ':LABEL'] # Định dạng chuẩn cho Neo4j CSV Header (Node)

# Lưu file Nodes
NODES_CSV_PATH = '/content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/neo4j_nodes.csv'
nodes_df.to_csv(NODES_CSV_PATH, index=False, encoding='utf-8-sig')

print("\n--- Định dạng Neo4j Nodes ---")
print(f"Số lượng Nodes độc nhất: {len(nodes_df)}")
print(f"🎉 Đã lưu file Nodes thành công tới: {NODES_CSV_PATH}")
print(nodes_df.head())


--- Định dạng Neo4j Nodes ---
Số lượng Nodes độc nhất: 4525
🎉 Đã lưu file Nodes thành công tới: /content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/neo4j_nodes.csv
              name:ID         :LABEL
0          VIÊM GAN B  DISEASESYMTOM
1  XUNG HUYẾT HANG VỊ  DISEASESYMTOM
2        CAO HUYẾT ÁP  DISEASESYMTOM
3            MÃN TÍNH  DISEASESYMTOM
4              AMIDAN  DISEASESYMTOM


In [10]:
# Chuẩn bị DataFrame Relationships
relationships_df = df_final[['head_entity_text_canonical',
                              'relation_type',
                              'tail_entity_text_canonical',
                              'confidence']].copy()

# Đổi tên cột sang định dạng chuẩn cho Neo4j CSV Header (Relationship)
relationships_df.columns = [':START_ID', ':TYPE', ':END_ID', 'confidence']

# Lưu file Relationships
RELATIONSHIPS_CSV_PATH = '/content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/neo4j_relationships.csv'
relationships_df.to_csv(RELATIONSHIPS_CSV_PATH, index=False, encoding='utf-8-sig')

print("\n--- Định dạng Neo4j Relationships ---")
print(f"Số lượng Relationships: {len(relationships_df)}")
print(f"🎉 Đã lưu file Relationships thành công tới: {RELATIONSHIPS_CSV_PATH}")
print(relationships_df.head())


--- Định dạng Neo4j Relationships ---
Số lượng Relationships: 30955
🎉 Đã lưu file Relationships thành công tới: /content/drive/MyDrive/DO_AN_DS_KI_1_NAM_3/DS201/NLe/KG-building/neo4j_relationships.csv
            :START_ID        :TYPE             :END_ID  confidence
0          VIÊM GAN B  HAS_SYMPTOM   TRÀO NGƯỢC DẠ DÀY    0.999892
1          VIÊM GAN B  HAS_SYMPTOM  XUNG HUYẾT HANG VỊ    0.999892
2  XUNG HUYẾT HANG VỊ  HAS_SYMPTOM   TRÀO NGƯỢC DẠ DÀY    0.999892
3        CAO HUYẾT ÁP  HAS_SYMPTOM          VIÊM GAN C    0.999892
4            MÃN TÍNH  HAS_SYMPTOM          NỔI MỀ ĐAY    0.999892
